In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## Image segmentation with SAM 3

This notebook demonstrates how to use SAM 3 for image segmentation with text or visual prompts. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "face")
- **Box prompts**: Using bounding boxes as exemplar visual prompts

In [ ]:
using_colab = False

In [ ]:
import os
import matplotlib.pyplot as plt

import mlx_sam3p1
from PIL import Image
from mlx_sam3p1 import build_sam3_image_model
from mlx_sam3p1.model.box_ops import box_xywh_to_cxcywh
from mlx_sam3p1.model.sam3_image_processor import Sam3Processor
from examples.visualization import draw_box_on_image, normalize_bbox, plot_results

package_root = os.path.dirname(mlx_sam3p1.__file__)
repo_root = os.path.dirname(package_root)

In [ ]:
import mlx.core as mx

In [ ]:
bpe_path = f"{package_root}/assets/bpe_simple_vocab_16e6.txt.gz"
checkpoint_path = os.environ.get("MLX_SAM3P1_CHECKPOINT")
model = build_sam3_image_model(
    bpe_path=bpe_path,
    checkpoint_path=checkpoint_path
)

In [ ]:
image_path = os.environ.get("MLX_SAM3P1_EXAMPLE_IMAGE")
if image_path is None:
    raise RuntimeError("Set MLX_SAM3P1_EXAMPLE_IMAGE to an image file path")
image = Image.open(image_path)
width, height = image.size
processor = Sam3Processor(model, confidence_threshold=0.5)
inference_state = processor.set_image(image)

In [ ]:
processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(state=inference_state, prompt="faces")

img0 = Image.open(image_path)
plot_results(img0, inference_state)

### Visual prompt: a single bounding box

In [ ]:
# Here the box is in  (x,y,w,h) format, where (x,y) is the top left corner.
box_input_xywh = mx.array([480.0, 290.0, 110.0, 360.0]).reshape(-1, 4)
box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)

norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()
print("Normalized box input:", norm_box_cxcywh)

processor.reset_all_prompts(inference_state)
inference_state = processor.add_geometric_prompt(
    state=inference_state, box=norm_box_cxcywh, label=True
)

img0 = Image.open(image_path)
image_with_box = draw_box_on_image(img0, box_input_xywh.flatten().tolist())
plt.imshow(image_with_box)
plt.axis("off")  # Hide the axis
plt.show()

In [ ]:
plot_results(img0, inference_state)

### Visual prompt: multi-box prompting (with positive and negative boxes)

In [ ]:
box_input_xywh = [[480.0, 290.0, 110.0, 360.0], [370.0, 280.0, 115.0, 375.0]]
box_input_cxcywh = box_xywh_to_cxcywh(mx.array(box_input_xywh).reshape(-1,4))
norm_boxes_cxcywh = normalize_bbox(box_input_cxcywh, width, height).tolist()

box_labels = [True, False]

processor.reset_all_prompts(inference_state)

for box, label in zip(norm_boxes_cxcywh, box_labels):
    inference_state = processor.add_geometric_prompt(
        state=inference_state, box=box, label=label
    )

img0 = Image.open(image_path)
image_with_box = img0
for i in range(len(box_input_xywh)):
    if box_labels[i] == 1:
        color = (0, 255, 0)
    else:
        color = (255, 0, 0)
    image_with_box = draw_box_on_image(image_with_box, box_input_xywh[i], color)
plt.imshow(image_with_box)
plt.axis("off")  # Hide the axis
plt.show()

In [ ]:
plot_results(img0, inference_state)